# Label-swap test on PDEBench Burgers nu = 0.001: one seed's trainings (Kaggle CPU)

Protocol `tools/fieldlint/bench/protocol_labelswap_burgers.md` (section 3.1). Runs `scripts/labelswap_burgers.py train --seeds SEED --threads 4`: 5 folds x 2 arms. CPU only, no accelerator. Attach the private dataset `rajulkabir/thermo-labelswap-burgers` (flat: file names are repo paths with `--` for `/`; SOURCE.txt gives the base commit and sha256 of every file). The dataset version must include `results/labelswap_burgers/labels.npz`.
Output: `/kaggle/working/predictions_s{SEED}.npz` and `session_s{SEED}.json`.

In [ ]:
SEED = 0   # one session per seed: 0, 1, 2
import glob, json, os, platform, shutil, subprocess, sys, time
src = glob.glob('/kaggle/input/**/SOURCE.txt', recursive=True)
assert len(src) == 1, glob.glob('/kaggle/input/**', recursive=True)[:30]
W = '/kaggle/working/tf'
import zipfile
def put(name, data):                                     # file names are repo paths with '--' for '/'
    dst = os.path.join(W, *name.split('--'))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    open(dst, 'wb').write(data)
n = 0
for root, _, fs in os.walk(os.path.dirname(src[0])):     # bundle.zip, or its contents if Kaggle unpacked it
    for f in fs:
        p = os.path.join(root, f)
        if f.endswith('.zip'):
            with zipfile.ZipFile(p) as z:
                for m in z.namelist():
                    put(m, z.read(m)); n += 1
        elif '--' in f:
            put(f, open(p, 'rb').read()); n += 1
import hashlib
want = [l.split('  ') for l in open(src[0]).read().split('sha256:')[1].split(chr(10)) if l.strip()]
bad = [r for h, r in want if hashlib.sha256(open(os.path.join(W, r), 'rb').read()).hexdigest() != h]
assert n == len(want) and not bad, (n, len(want), bad)
print(n, 'files restored, sha256 match SOURCE.txt')
print(open(src[0]).read())
import numpy
cpu = [l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')]
info = {'numpy': numpy.__version__, 'python': platform.python_version(), 'cpu': cpu[0] if cpu else '', 'cores': os.cpu_count()}
import torch
info.update(seed=SEED, torch=torch.__version__)
print(info)
t = time.time()
r = subprocess.run([sys.executable, 'scripts/labelswap_burgers.py', 'train', '--seeds', str(SEED), '--threads', '4'], cwd=W)
info['seconds'], info['returncode'] = round(time.time() - t), r.returncode
shutil.copy(f'{W}/results/labelswap_burgers/predictions.partial.npz', f'/kaggle/working/predictions_s{SEED}.npz')
json.dump(info, open(f'/kaggle/working/session_s{SEED}.json', 'w'), indent=1)
print(info)
assert r.returncode == 0
